In [1]:
# notebook : semdetect_repro.ipynb — cellule 1 : feature store sur le test Sem-Detect
import logging
import os
from pathlib import Path

import numpy as np
import pandas as pd

from src.config import PROJ_ROOT
from src.data.dataset_semdetect import SemDetectDataset
from src.detector.sem_detect.feature_store import SemDetectFeatureStore
from src.detector.sem_detect.semdetect.classifier import SemDetectClassifier

logging.getLogger("httpx").setLevel(logging.WARNING)
DEVICE = "cuda:0"
REVISION = "18b74b551670bff6b04e57570232a1694997863b"

clf = SemDetectClassifier(str(PROJ_ROOT / "src/detector/sem_detect/models/lightgbm_best_model.pkl"))

te = SemDetectDataset(config={
    "name": "sem_detect", "repo_id": "Sem-Detect/ML_Conferences-Peer-Reviews", "revision": REVISION,
    "split": "test", "conference": None, "date": None, "generator": None,
    "label_mode": "three_class", "balance": "none", "subset_size": None,
    "val_fraction": None, "val_seed": 0,
    "no_duplicate": True, "keep_meta": True, "with_paper": False, "keep_claims": True,
    "exclude_ids_path": None, "seed": 0,
}).to_pandas()

store = SemDetectFeatureStore(
    root=Path(os.environ["CACHE_DIR"]) / "sem_detect_store",
    claims_source=f"hf:Sem-Detect/ML_Conferences-Peer-Reviews@{REVISION}",
    device=DEVICE,
)
store.ensure_embeddings(te)   # quelques minutes
store.ensure_textual(te)      # ~1-2 h ; relancer la cellule après interruption reprend où elle s'était arrêtée
print(store.dir)

/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-01 13:02:59.571 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: /home/cyaiche/generated_text_detection_hub
INFO:src.detector.sem_detect.semdetect.classifier:Loading Sem-Detect model from /home/cyaiche/generated_text_detection_hub/src/detector/sem_detect/models/lightgbm_best_model.pkl
/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
 

/data2/cyaiche/cache_dir/sem_detect_store/2791ce73dd94


In [2]:
# notebook : semdetect_repro.ipynb — cellule 2 : protocole du papier, 3 classes + binaire
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score

from src.detector.sem_detect.references import build_feature_matrix, paper_protocol_references, predict_proba
from src.evaluation.metrics import get_tpr_at_fpr

refs = paper_protocol_references(te, k=3, seed=0)
X, kept, skipped = build_feature_matrix(store, te["review-id"].tolist(), refs, clf.similarity_threshold)
P = predict_proba(clf, X)

meta = te.set_index("review-id").loc[kept]
print(f"classées : {len(kept)} / {len(te)}")
print(pd.Series(skipped).rename("raison").to_frame()
      .join(te.set_index("review-id")["class"]).value_counts().to_string(), "\n")

# 3 classes, ordre et noms de scripts/evaluate.py (indice 0 = IA)
CLASS_TO_IDX = {"ai": 0, "rewrite": 1, "human": 2}
names = ["AI", "LLM-Refined", "Human"]
y_true = meta["class"].map(CLASS_TO_IDX).to_numpy()
y_pred = P.argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=names, digits=3))
print(f"Macro-F1 : {f1_score(y_true, y_pred, average='macro'):.4f}   (papier : 0.84)\n")
print("Confusion (% par ligne vraie) — papier Fig.5 : AI [91.18 8.53 0.29] | "
      "Refined [3.44 91.61 4.95] | Human [0.66 35.38 63.95]")
print(pd.DataFrame(np.round(confusion_matrix(y_true, y_pred, normalize="true") * 100, 2),
                   index=names, columns=names), "\n")

# binaire humain vs IA (Table 1) et effondré humain+réécriture vs IA (Fig. 4) ; score = P(ai)
p_ai = P[:, 0]
is_ai = (meta["class"] == "ai").to_numpy()
m = meta["class"].isin(["human", "ai"]).to_numpy()
print(f"Humain vs IA          AUC={roc_auc_score(is_ai[m], p_ai[m]):.4f}  "
      f"TPR@1%={get_tpr_at_fpr(is_ai[m], p_ai[m], 0.01):.3f}  TPR@0.1%={get_tpr_at_fpr(is_ai[m], p_ai[m], 0.001):.3f}"
      f"   (papier : 0.999 / 0.973 / 0.760)")
print(f"(Humain+Réécr.) vs IA AUC={roc_auc_score(is_ai, p_ai):.4f}   (papier : 0.990)")

classées : 4373 / 4410
raison                        class  
cible < 4 claims sémantiques  rewrite    30
                              human       7 

              precision    recall  f1-score   support

          AI      0.938     0.911     0.925      1364
 LLM-Refined      0.873     0.923     0.897      2410
       Human      0.775     0.644     0.704       599

    accuracy                          0.881      4373
   macro avg      0.862     0.826     0.842      4373
weighted avg      0.880     0.881     0.879      4373

Macro-F1 : 0.8418   (papier : 0.84)

Confusion (% par ligne vraie) — papier Fig.5 : AI [91.18 8.53 0.29] | Refined [3.44 91.61 4.95] | Human [0.66 35.38 63.95]
                AI  LLM-Refined  Human
AI           91.13         8.58   0.29
LLM-Refined   3.20        92.32   4.48
Human         0.83        34.72  64.44 

Humain vs IA          AUC=0.9980  TPR@1%=0.928  TPR@0.1%=0.674   (papier : 0.999 / 0.973 / 0.760)
(Humain+Réécr.) vs IA AUC=0.9912   (papier : 0.990)


In [3]:
# notebook : semdetect_repro.ipynb — cellule 3 : protocole du script officiel + choix du score binaire
from src.detector.sem_detect.references import _stable_unit

DEFAULT_AI_AUTHORS = ["gemini-2.5-flash", "gemini-2.5-pro", "deepseek.v3-v1:0", "qwen.qwen3-235b-a22b-2507-v1:0"]


def script_protocol_references(df: pd.DataFrame, seed: int) -> dict:
    """Émulation de scripts/evaluate.py : UNE review IA par modèle et par papier, note ignorée,
    modèle de la cible non exclu. Le « premier » du script dépend de l'ordre du JSON, inconnu :
    tirage déterministe, répété sur plusieurs graines."""
    ai = df[df["class"] == "ai"]
    first = {(sub, gen): min(g["review-id"], key=lambda r: _stable_unit(r, seed))
             for (sub, gen), g in ai.groupby(["submission_id", "generator"])}
    return {rid: [first[(sub, a)] for a in DEFAULT_AI_AUTHORS if (sub, a) in first]
            for rid, sub in zip(df["review-id"], df["submission_id"])}


rows = []
for seed in range(5):
    r = script_protocol_references(te, seed)
    Xs, kept_s, _ = build_feature_matrix(store, te["review-id"].tolist(), r, clf.similarity_threshold)
    Ps = predict_proba(clf, Xs)
    ms = te.set_index("review-id").loc[kept_s]
    yt, yp = ms["class"].map(CLASS_TO_IDX).to_numpy(), Ps.argmax(axis=1)
    ai_mask = ms["class"].eq("ai").to_numpy()
    self_ref = np.array([rid in r[rid] for rid in kept_s])
    rows.append({
        "graine": seed,
        "macro_f1": f1_score(yt, yp, average="macro"),
        "rappel_ia": (yp[ai_mask] == 0).mean(),
        "part_ia_auto_référencée": self_ref[ai_mask].mean(),
        "rappel_ia | auto-réf": (yp[ai_mask & self_ref] == 0).mean(),
        "rappel_ia | sans": (yp[ai_mask & ~self_ref] == 0).mean(),
    })
print("Protocole du script (papier : macro-F1 0.8418, rappel IA 0.911)")
print(pd.DataFrame(rows).round(4).to_string(index=False), "\n")

# score binaire : P(ai) contre 1 - P(human), sur le protocole du papier
for name, s in {"P(ai)": P[:, 0], "1 - P(human)": 1 - P[:, 2]}.items():
    print(f"{name:<13} humain vs IA : AUC={roc_auc_score(is_ai[m], s[m]):.4f}  TPR@1%={get_tpr_at_fpr(is_ai[m], s[m], 0.01):.3f} | "
          f"(humain+réécr.) vs IA : AUC={roc_auc_score(is_ai, s):.4f}")

Protocole du script (papier : macro-F1 0.8418, rappel IA 0.911)
 graine  macro_f1  rappel_ia  part_ia_auto_référencée  rappel_ia | auto-réf  rappel_ia | sans
      0    0.8503     0.9919                   0.4663                   1.0            0.9849
      1    0.8514     0.9883                   0.4663                   1.0            0.9780
      2    0.8532     0.9883                   0.4663                   1.0            0.9780
      3    0.8519     0.9868                   0.4663                   1.0            0.9753
      4    0.8506     0.9897                   0.4663                   1.0            0.9808 

P(ai)         humain vs IA : AUC=0.9980  TPR@1%=0.928 | (humain+réécr.) vs IA : AUC=0.9912
1 - P(human)  humain vs IA : AUC=0.9951  TPR@1%=0.917 | (humain+réécr.) vs IA : AUC=0.9127
